In [1]:
import numpy as np
import random as rd
import copy
import psutil
import gc
import pickle
from qiskit.quantum_info import SparsePauliOp
n_qubit = 1
dim     = 2**n_qubit
nld     = 11
lds     = np.linspace(0,1,num=nld)

In [2]:
def memory_usage(message: str = 'debug'):
    # this memory_usage function is imported from https://jybaek.tistory.com/895
    # current process RAM usage
    p = psutil.Process()
    rss = p.memory_info().rss / 2 ** 30 # Bytes to GiB
    print(f"[{message}] memory usage: {rss: 10.5f} GiB")

In [3]:
t_x          = 0.5
t_zs         = []
hamiltonians = []
alpha        = 0
for ld in lds:
    t_z = -1 + 2.0 * ld
    h = SparsePauliOp.from_list([('X',t_x),('Z',t_z)])
    t_zs.append(t_z)
    hamiltonians.append(h)
    print(hamiltonians[alpha])
    alpha += 1


SparsePauliOp(['X', 'Z'],
              coeffs=[ 0.5+0.j, -1. +0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[ 0.5+0.j, -0.8+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[ 0.5+0.j, -0.6+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[ 0.5+0.j, -0.4+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[ 0.5+0.j, -0.2+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[0.5+0.j, 0. +0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[0.5+0.j, 0.2+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[0.5+0.j, 0.4+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[0.5+0.j, 0.6+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[0.5+0.j, 0.8+0.j])
SparsePauliOp(['X', 'Z'],
              coeffs=[0.5+0.j, 1. +0.j])


In [4]:
hamiltonian_diffs = []
for ild in range(nld-1):
    hamiltonian_diffs.append((hamiltonians[ild+1]-hamiltonians[ild]).simplify())
    print(hamiltonian_diffs[ild])

SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])
SparsePauliOp(['Z'],
              coeffs=[0.2+0.j])


In [5]:
hamiltonian_diffs_list = []
for ild in range(nld-1):
    hamiltonian_diffs_list.append(hamiltonian_diffs[ild].to_list())
    print(hamiltonian_diffs_list[ild])

[('Z', (0.19999999999999996+0j))]
[('Z', (0.20000000000000007+0j))]
[('Z', (0.20000000000000007+0j))]
[('Z', (0.19999999999999996+0j))]
[('Z', (0.19999999999999996+0j))]
[('Z', (0.20000000000000018+0j))]
[('Z', (0.19999999999999996+0j))]
[('Z', (0.19999999999999996+0j))]
[('Z', (0.19999999999999996+0j))]
[('Z', (0.19999999999999996+0j))]


In [6]:
n_hamiltonians = len(hamiltonians)

In [7]:
# exact eigenvalues
eigen_energies_exact  = np.zeros((nld,dim),dtype=float)
eigen_vectors_exact   = np.zeros((nld,dim,dim),dtype=complex)
for ild in range(nld):
    eigen_e, eigen_v = np.linalg.eigh(hamiltonians[ild].to_matrix())
    indx = np.argsort(eigen_e.real)
    for i in range(dim):
        eigen_energies_exact[ild,i]   = eigen_e[indx[i]].real
        eigen_vectors_exact[ild,:,i] = eigen_v[:,indx[i]]


In [8]:
x_op = SparsePauliOp.from_list([('X',1.0)])
y_op = SparsePauliOp.from_list([('Y',1.0)])
z_op = SparsePauliOp.from_list([('Z',1.0)])
x_mat = x_op.to_matrix()
y_mat = y_op.to_matrix()
z_mat = z_op.to_matrix()

In [9]:
def ComputeUnitaryParams(U):
    theta = 2.0 * np.arccos(np.abs(U[0,0]))
    if (theta<1E-6):
        theta = 2.0*np.arcsin(np.abs(U[0,1]))
    gamma = np.angle(U[0,0])
    if (theta<1E-10):
        phi   = np.angle(U[1,1]/U[0,0])
        lam   = 0
    else:
        phi   = np.angle(U[1,0]/np.sin(theta/2)) - gamma
        lam   = np.angle(-U[0,1]/np.sin(theta/2)) - gamma
    return [theta, phi, lam, gamma]

def ExactTimeEvolution (alpha, time):
    Vl = copy.deepcopy(eigen_vectors_exact[alpha,:,:])
    evol = np.zeros((dim,dim),dtype=complex)
    exp_d = np.diag(np.exp(-1j*eigen_energies_exact[alpha,:]*time))
    evol = Vl@exp_d@Vl.conj().T
    return evol

def TrotterTimeEvolution (alpha, Nt, time):
    dtime  = time/Nt
    dtx = dtime * t_x
    dtz = dtime * t_zs[alpha]
    evol = np.identity(dim)
    for it in range(Nt):
        evol_X = np.cos(dtx)*np.identity(dim) - 1j*np.sin(dtx) * x_mat
        evol_Z = np.cos(dtz)*np.identity(dim) - 1j*np.sin(dtz) * z_mat
        evol = evol_Z@evol
        evol = evol_X@evol
    return evol

In [10]:
# exact results
norms_exact  = np.ones((nld,dim),dtype=float)
for i in range(dim):
    phi = eigen_vectors_exact[0,:,i]
    for alpha in range(1,nld):
        proj_matrix = np.outer(eigen_vectors_exact[alpha,:,i],eigen_vectors_exact[alpha,:,i].conj())
        phi = proj_matrix@phi
        norms_exact[alpha,i] = phi.conj()@phi

x_exps_exact = np.zeros((nld,dim),dtype=float)
y_exps_exact = np.zeros((nld,dim),dtype=float)
z_exps_exact = np.zeros((nld,dim),dtype=float)
for ild in range(nld):
    for i in range(dim):
        x_exps_exact[ild,i] = eigen_vectors_exact[ild,:,i].conj().transpose()@x_mat@eigen_vectors_exact[ild,:,i]
        y_exps_exact[ild,i] = eigen_vectors_exact[ild,:,i].conj().transpose()@y_mat@eigen_vectors_exact[ild,:,i]
        z_exps_exact[ild,i] = eigen_vectors_exact[ild,:,i].conj().transpose()@z_mat@eigen_vectors_exact[ild,:,i]

/tmp/ipykernel_53114/1490341378.py:8: ComplexWarning: Casting complex values to real discards the imaginary part
  norms_exact[alpha,i] = phi.conj()@phi
/tmp/ipykernel_53114/1490341378.py:15: ComplexWarning: Casting complex values to real discards the imaginary part
  x_exps_exact[ild,i] = eigen_vectors_exact[ild,:,i].conj().transpose()@x_mat@eigen_vectors_exact[ild,:,i]
/tmp/ipykernel_53114/1490341378.py:16: ComplexWarning: Casting complex values to real discards the imaginary part
  y_exps_exact[ild,i] = eigen_vectors_exact[ild,:,i].conj().transpose()@y_mat@eigen_vectors_exact[ild,:,i]
/tmp/ipykernel_53114/1490341378.py:17: ComplexWarning: Casting complex values to real discards the imaginary part
  z_exps_exact[ild,i] = eigen_vectors_exact[ild,:,i].conj().transpose()@z_mat@eigen_vectors_exact[ild,:,i]


In [11]:
nmc = int(400)

In [12]:
theta_init = np.zeros((dim),dtype=float)
for i in range(dim):
    theta_init[i] = 2.0 * np.angle(eigen_vectors_exact[0,0,i]+ 1j*eigen_vectors_exact[0,1,i])

In [13]:
def Circuit_Initialize(circuit_, i_, qr_):
    # initialize qr_[1:] to i_ th eigenvector
    circuit_.ry(theta_init[i_],qr_[1])

In [14]:
import time as time_lib

In [15]:
from qiskit import transpile, Aer
from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit_aer.primitives import Sampler as AerSampler

estimator = AerEstimator()

nshot    = 4000

In [16]:
q_layout = [0,1]
z_hadamard_test = SparsePauliOp.from_sparse_list([('Z',[q_layout[0]],1)],num_qubits=n_qubit+1)
print(z_hadamard_test)

SparsePauliOp(['IZ'],
              coeffs=[1.+0.j])


In [17]:
from qiskit import QuantumRegister
from qiskit import QuantumCircuit, transpile

qr = QuantumRegister(n_qubit+1, 'q')

In [18]:
# parametrized circuit
from qiskit.circuit import ParameterVector
params = ParameterVector('θ',5)

In [19]:
#beta_list = [0.1, 0.25, 0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 20.0, 30.0, 40.0, 50.0]
#beta_list = [0.1, 0.25, 0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 12.0, 14.0, 16.0, 18.0, 20.0]
beta_list = [0.1, 0.25, 0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 
             11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0]
n_beta = len(beta_list)
print(n_beta)

23


In [20]:
# run qzmc;

norms_beta               = np.ones((n_beta,n_hamiltonians),dtype=float)
eigen_energies_beta      = np.zeros((n_beta,n_hamiltonians),dtype=float)
eigen_energies_beta[:,0] = eigen_energies_exact[0,0]

In [21]:
result_values_beta   = [[[] for _ in range(n_hamiltonians)] for _ in range(n_beta)]

In [22]:
memory_usage('before run')

[before run] memory usage:    0.17332 GiB


In [23]:
# observable amplitudes
n_obs = 3
# 3 different observables are computed
# 0; norm, 1; dE1, 2; Z

In [24]:
beta_0 = 5.0

In [25]:
with open('MDH.time.binary','rb') as file_:
    [O_timelists] = pickle.load(file_)

In [26]:
i = 0 # only consider the ground state
for i_beta in range(n_beta):
    beta = beta_list[i_beta]
    # real part measurement
    circuit_real = QuantumCircuit(qr,name='-')
    Circuit_Initialize(circuit_real,i,qr)
    circuit_real.h(qr[0])
    circuit_real.cu(params[0],params[1],params[2],params[3],qr[0],qr[1])
    circuit_real.p(params[4],qr[0])
    circuit_real.h(qr[0])

    eps = eigen_vectors_exact[0,:,i].conj().T@hamiltonians[1].to_matrix()@eigen_vectors_exact[0,:,i]
    eps = eps.real
    for alpha in range(1,n_hamiltonians):
        params_list = []
        # 0; norm measurement
        i_obs = 0
        for imc in range(nmc):
            U_evol = np.identity(dim,dtype=complex)
            times  = np.array(O_timelists[i][alpha][i_obs][imc]) * (beta/beta_0)
            phase  = 0.0
            i_time = 0
            # construction of P
            for alpha_ in range(1,alpha):
                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                phase += eigen_energies_beta[i_beta,alpha_] * time
                i_time += 1

            time = times[i_time]
            U_evol = ExactTimeEvolution(alpha,time)@U_evol
            phase += eps * time
            i_time += 1

            # construction of P^\dagger

            time = times[i_time]
            U_evol = ExactTimeEvolution(alpha,time)@U_evol
            phase += eps * time
            i_time += 1

            for alpha_ in reversed(range(1,alpha)):
                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                phase += eigen_energies_beta[i_beta,alpha_] * time
                i_time += 1

            theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
            params_list.append([theta,phi,lam,gamma,phase])
        # 1; dE1 measurement
        i_obs = 1
        nhd1 = len(hamiltonian_diffs[alpha-1])
        for ihd in range(nhd1):
            # pass for constant contribution
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                continue
            for imc in range(nmc):
                U_evol = np.identity(dim,dtype=complex)
                times  = O_timelists[i][alpha][i_obs][imc]
                phase  = 0.0
                i_time = 0
                # construction of P
                for alpha_ in range(1,alpha):
                    time = times[i_time]
                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                    phase += eigen_energies_beta[i_beta,alpha_] * time
                    i_time += 1

                # insertion of (H[alpha]-H[alpha-1])
                Mat = hamiltonian_diffs[alpha-1].paulis[ihd].to_matrix()

                U_evol = Mat@U_evol

                # construction of P; continued
                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha,time)@U_evol
                phase += eps * time
                i_time += 1

                # construction of P^\dagger

                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha,time)@U_evol
                phase += eps * time
                i_time += 1

                for alpha_ in reversed(range(1,alpha)):
                    time = times[i_time]
                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                    phase += eigen_energies_beta[i_beta,alpha_] * time
                    i_time += 1
                theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
                params_list.append([theta,phi,lam,gamma,phase])

        # 2; dE2 measurement
        i_obs = 2
        if (alpha<n_hamiltonians-1):
            nhd2 = len(hamiltonian_diffs[alpha])
        else:
            nhd2 = 0
        for ihd in range(nhd2):
            # pass for constant contribution
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                continue
            for imc in range(nmc):
                U_evol = np.identity(dim,dtype=complex)
                times  = O_timelists[i][alpha][i_obs][imc]
                phase  = 0.0
                i_time = 0
                # construction of P
                for alpha_ in range(1,alpha):
                    time = times[i_time]
                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                    phase += eigen_energies_beta[i_beta,alpha_] * time
                    i_time += 1

                # construction of P; continued
                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha,time)@U_evol
                phase += eps * time
                i_time += 1

                # insertion of (H[alpha+1]-H[alpha])
                Mat = hamiltonian_diffs[alpha].paulis[ihd].to_matrix()

                U_evol = Mat@U_evol

                # construction of P^\dagger

                time = times[i_time]
                U_evol = ExactTimeEvolution(alpha,time)@U_evol
                phase += eps * time
                i_time += 1

                for alpha_ in reversed(range(1,alpha)):
                    time = times[i_time]
                    U_evol = ExactTimeEvolution(alpha_,time)@U_evol
                    phase += eigen_energies_beta[i_beta,alpha_] * time
                    i_time += 1
                theta, phi, lam, gamma = ComputeUnitaryParams(U_evol)
                params_list.append([theta,phi,lam,gamma,phase])

        circuits = [circuit_real.assign_parameters({params: params_}) for params_ in params_list]

        z_hadamards = [z_hadamard_test]*len(circuits)
        estimator = AerEstimator(run_options={"shots": 4000})
        job = estimator.run(circuits,z_hadamards)
        result_values = job.result().values
        result_values_beta[i_beta][alpha] = result_values
        # compute values
        norm    = 0.0
        dE1     = 0.0
        dE2     = 0.0

        i_meas = 0
        # 0; norm
        for imc in range(nmc):
            norm += result_values_beta[i_beta][alpha][i_meas]
            i_meas += 1
        # 1; dE1
        nhd1 = len(hamiltonian_diffs[alpha-1])
        for ihd in range(nhd1):
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                continue
            coeff = hamiltonian_diffs_list[alpha-1][ihd][1]
            for imc in range(nmc):
                dE1 +=  coeff * result_values_beta[i_beta][alpha][i_meas]
                i_meas += 1

        # 2; dE2
        if (alpha<n_hamiltonians-1):
            nhd2 = len(hamiltonian_diffs[alpha])
        else:
            nhd2 = 0
        for ihd in range(nhd2):
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                continue
            coeff = hamiltonian_diffs_list[alpha][ihd][1]
            for imc in range(nmc):
                dE2 += coeff * result_values_beta[i_beta][alpha][i_meas]
                i_meas += 1

        dE1     /=norm
        dE2     /=norm

        norm    /=nmc

        # add constant contributions
        for ihd in range(nhd1):
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                dE1 += hamiltonian_diffs_list[alpha-1][ihd][1]

        for ihd in range(nhd2):
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                dE2 += hamiltonian_diffs_list[alpha][ihd][1]

        eigen_energies_beta[i_beta,alpha] = eigen_energies_beta[i_beta,alpha-1] + dE1
        norms_beta[i_beta,alpha] = norm

        if (alpha<n_hamiltonians-1):
            eps = eigen_energies_beta[i_beta,alpha] + dE2
            eps = eps.real

        print(alpha, norms_beta[i_beta,alpha], eigen_energies_beta[i_beta,alpha]-eigen_energies_exact[alpha,i])
        if (alpha<n_hamiltonians-1):
            print(alpha, eps, eigen_energies_exact[alpha+1,i])
        st = '# {i_beta}/{n_beta}: {percent}%'.format(i_beta=i_beta+1,n_beta=n_beta,percent=((alpha)/(n_hamiltonians-1)*100))

        del job
        del z_hadamards
        del circuits 
        del estimator
        collected = gc.collect()
        memory_usage(st)




/tmp/ipykernel_53114/699500863.py:197: ComplexWarning: Casting complex values to real discards the imaginary part
  eigen_energies_beta[i_beta,alpha] = eigen_energies_beta[i_beta,alpha-1] + dE1


1 0.9999000000000012 -3.316527320773588e-05
1 -0.7744088762386443 -0.7810249675906655
[# 1/23: 10.0%] memory usage:    0.25153 GiB
2 0.9996487500000015 -0.001290719036564747
2 -0.6297393441869483 -0.6403124237432848
[# 1/23: 20.0%] memory usage:    0.27786 GiB
3 0.9987887500000013 -0.003931273186424611
3 -0.5229993397020174 -0.5385164807134504
[# 1/23: 30.0%] memory usage:    0.27988 GiB
4 0.9978037500000011 -0.005282865310582641
4 -0.4740025548213542 -0.5
[# 1/23: 40.0%] memory usage:    0.27993 GiB
5 0.9950750000000019 -0.009085128502740791
5 -0.5083701070219478 -0.5385164807134505
[# 1/23: 50.0%] memory usage:    0.28092 GiB
6 0.9897725000000009 -0.005866153024079113
6 -0.6125755770654154 -0.640312423743285
[# 1/23: 60.0%] memory usage:    0.28000 GiB
7 0.9811112500000008 -0.0013239617986398056
7 -0.7555391157368949 -0.7810249675906655
[# 1/23: 70.0%] memory usage:    0.28097 GiB
8 0.9627374999999999 0.014493426307949586
8 -0.9047000451583829 -0.9433981132056604
[# 1/23: 80.0%] memo

In [27]:
#with open('MDH.beta.results','rb') as file_:
#    [result_values_beta] = pickle.load(file_)

with open('MDH.beta.results','wb') as file_:
    pickle.dump([result_values_beta],file_)


In [28]:
norms_raw_beta            = np.zeros((n_beta,n_hamiltonians,nmc),dtype=float)
dEnorm_raw_beta           = np.zeros((n_beta,n_hamiltonians,nmc),dtype=float)

In [29]:
i = 0 # only consider the ground state
for i_beta in range(n_beta):
    beta = beta_list[i_beta]
    eps = eigen_vectors_exact[0,:,i].conj().T@hamiltonians[1].to_matrix()@eigen_vectors_exact[0,:,i]
    eps = eps.real
    for alpha in range(1,n_hamiltonians):
        # compute values
        norm    = 0.0
        dE1     = 0.0
        dE2     = 0.0

        i_meas = 0
        # 0; norm
        for imc in range(nmc):
            norms_raw_beta[i_beta][alpha][imc] = result_values_beta[i_beta][alpha][i_meas]
            norm += norms_raw_beta[i_beta][alpha][imc]
            i_meas += 1
        # 1; dE1
        nhd1 = len(hamiltonian_diffs[alpha-1])
        for ihd in range(nhd1):
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                continue
            coeff = hamiltonian_diffs_list[alpha-1][ihd][1]
            for imc in range(nmc):
                dEnorm_raw_beta[i_beta][alpha][imc] +=  coeff * result_values_beta[i_beta][alpha][i_meas]
                i_meas += 1
        for imc in range(nmc):
            dE1 += dEnorm_raw_beta[i_beta][alpha][imc]

        # 2; dE2
        if (alpha<n_hamiltonians-1):
            nhd2 = len(hamiltonian_diffs[alpha])
        else:
            nhd2 = 0
        for ihd in range(nhd2):
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                continue
            coeff = hamiltonian_diffs_list[alpha][ihd][1]
            for imc in range(nmc):
                dE2 += coeff * result_values_beta[i_beta][alpha][i_meas]
                i_meas += 1

        dE1     /=norm
        dE2     /=norm
        norm    /=nmc

        # add constant contributions
        for ihd in range(nhd1):
            if (hamiltonian_diffs_list[alpha-1][ihd][0]=='I'*n_qubit):
                dE1 += hamiltonian_diffs_list[alpha-1][ihd][1]

        for ihd in range(nhd2):
            if (hamiltonian_diffs_list[alpha][ihd][0]=='I'*n_qubit):
                dE2 += hamiltonian_diffs_list[alpha][ihd][1]

        eigen_energies_beta[i_beta,alpha] = eigen_energies_beta[i_beta,alpha-1] + dE1
        norms_beta[i_beta,alpha] = norm

        if (alpha<n_hamiltonians-1):
            eps = eigen_energies_beta[i_beta,alpha] + dE2
            eps = eps.real

        print(alpha, norms_beta[i_beta,alpha], eigen_energies_beta[i_beta,alpha]-eigen_energies_exact[alpha,i])
        if (alpha<n_hamiltonians-1):
            print(alpha, eps, eigen_energies_exact[alpha+1,i])
        st = '# {i_beta}/{n_beta}: {percent}%'.format(i_beta=i_beta+1,n_beta=n_beta,percent=((alpha)/(n_hamiltonians-1)*100))

        #collected = gc.collect()
        memory_usage(st)




1 0.9999000000000012 -3.316527320773588e-05
1 -0.7744088762386443 -0.7810249675906655
[# 1/23: 10.0%] memory usage:    0.29475 GiB
2 0.9996487500000015 -0.001290719036564747
2 -0.6297393441869483 -0.6403124237432848
[# 1/23: 20.0%] memory usage:    0.29475 GiB
3 0.9987887500000013 -0.003931273186424611
3 -0.5229993397020174 -0.5385164807134504
[# 1/23: 30.0%] memory usage:    0.29475 GiB
4 0.9978037500000011 -0.005282865310582641
4 -0.4740025548213542 -0.5
[# 1/23: 40.0%] memory usage:    0.29475 GiB
5 0.9950750000000019 -0.009085128502740791
5 -0.5083701070219478 -0.5385164807134505
[# 1/23: 50.0%] memory usage:    0.29475 GiB
6 0.9897725000000009 -0.005866153024079113
6 -0.6125755770654154 -0.640312423743285
[# 1/23: 60.0%] memory usage:    0.29475 GiB
7 0.9811112500000008 -0.0013239617986398056
7 -0.7555391157368949 -0.7810249675906655
[# 1/23: 70.0%] memory usage:    0.29475 GiB
8 0.9627374999999999 0.014493426307949586
8 -0.9047000451583829 -0.9433981132056604
[# 1/23: 80.0%] memo

/tmp/ipykernel_53114/3865931032.py:25: ComplexWarning: Casting complex values to real discards the imaginary part
  dEnorm_raw_beta[i_beta][alpha][imc] +=  coeff * result_values_beta[i_beta][alpha][i_meas]


[# 14/23: 80.0%] memory usage:    0.29475 GiB
9 0.007711249999999995 -5.310103630147746
9 -11.484398160925137 -1.118033988749895
[# 14/23: 90.0%] memory usage:    0.29475 GiB
10 -0.0019550000000000097 -6.052347549999926
[# 14/23: 100.0%] memory usage:    0.29475 GiB
1 0.9948800000000009 0.0009569095152698326
1 -0.7726518823652058 -0.7810249675906655
[# 15/23: 10.0%] memory usage:    0.29475 GiB
2 0.9821362500000009 0.0021662098185475376
2 -0.6238166238523081 -0.6403124237432848
[# 15/23: 20.0%] memory usage:    0.29475 GiB
3 0.9446100000000002 0.006915814371035611
3 -0.5056332996465422 -0.5385164807134504
[# 15/23: 30.0%] memory usage:    0.29475 GiB
4 0.8413087500000007 0.02114505234639097
4 -0.4354142990729686 -0.5
[# 15/23: 40.0%] memory usage:    0.29475 GiB
5 0.4896025 0.04505319160526505
5 -0.4540246317310129 -0.5385164807134505
[# 15/23: 50.0%] memory usage:    0.29475 GiB
6 0.20873624999999996 -0.06223514117199358
6 -0.8648109311812661 -0.640312423743285
[# 15/23: 60.0%] memory

In [30]:
# compute standard deviations
import random as rd
std_norm    = np.zeros((n_beta,n_hamiltonians),dtype=float)
std_dE      = np.zeros((n_beta,n_hamiltonians),dtype=float)
std_E       = np.zeros((n_beta,n_hamiltonians),dtype=float)
n_boot = 1000
for i_beta in range(n_beta):
    for alpha in range(1,n_hamiltonians):
        norm_boot = np.zeros((n_boot),dtype=float)
        dE_boot   = np.zeros((n_boot),dtype=float)
        for i_boot in range(n_boot):
            norm_ = 0.0
            dE_   = 0.0
            for imc in range(nmc):
                jmc = rd.randrange(nmc)
                norm_ += norms_raw_beta[i_beta,alpha,jmc]
                dE_ += dEnorm_raw_beta[i_beta,alpha,jmc]

            dE_   = dE_/norm_
            norm_ = norm_/nmc
            norm_boot[i_boot] = norm_
            dE_boot[i_boot] = dE_
        norm_boot_mean = 0.0
        dE_boot_mean   = 0.0
        for i_boot in range(n_boot):
            norm_boot_mean += norm_boot[i_boot]
            dE_boot_mean   += dE_boot[i_boot]
        norm_boot_mean /= n_boot
        dE_boot_mean /= n_boot

        var_norm = 0.0
        var_dE = 0.0
        for i_boot in range(n_boot):
            var_norm += (norm_boot[i_boot]-norm_boot_mean)**2
            var_dE += (dE_boot[i_boot]-dE_boot_mean)**2
        var_norm /= (n_boot-1)
        var_dE    /= (n_boot-1)
        std_norm[i_beta,alpha] = np.sqrt(var_norm)
        std_dE[i_beta,alpha] = np.sqrt(var_dE)
        std_E[i_beta,alpha] = std_E[i_beta,alpha-1] + var_dE
for i_beta in range(n_beta):
    for alpha in range(n_hamiltonians):
        std_E[i_beta,alpha] = np.sqrt(std_E[i_beta,alpha])

In [31]:
for i_beta in range(n_beta):
    print(beta_list[i_beta],std_norm[i_beta,-1],std_E[i_beta,-1])

0.1 0.005949445630105425 0.006754609313437594
0.25 0.02137302369779802 0.012292390232929742
0.5 0.020609462255228608 0.019241065540161407
1.0 0.012259133767911936 0.011005499365898393
2.0 0.0068650346698836415 0.007569633343535356
3.0 0.006281147690831603 0.007452992179552401
4.0 0.0058665416039619155 0.007433914608183153
5.0 0.005965053842564085 0.007565521532944194
6.0 0.0062713693390337374 0.007487296405309198
7.0 0.005355487684646622 0.007631449741219377
8.0 0.006465183629409178 0.007700134920981892
9.0 0.007212817070369623 0.008146185628277933
10.0 0.026508062907978934 0.023116190697578528
11.0 0.03252817762583784 50.61925837929672
12.0 0.03072639713014809 15.729676476344412
13.0 0.029484187664225846 69.46431468273313
14.0 0.03307682207104303 120.95830137982983
15.0 0.0320837354431392 36.2326049974261
16.0 0.03151153390421602 21.1250379587616
17.0 0.03010261749549805 9.129649004622463
18.0 0.030083123143018777 21.40082065503805
19.0 0.03080968361301575 98.2560936345736
20.0 0.0309

In [32]:
# save to file
with open('beta.norm.E.save','w') as file_:
    s = '# beta , norm^2, std(norm^2), Enorm, std(Enorm) '
    s += '\n'
    file_.write(s)
    for i_beta in range(n_beta):
        beta = beta_list[i_beta]
        s = '{:.16e}'.format(beta)
        s += '  {:.16e}  {:.16e}'.format(norms_beta[i_beta,-1],std_norm[i_beta,-1])
        s += '  {:.16e}  {:.16e}'.format(eigen_energies_beta[i_beta,-1],std_E[i_beta,-1])
        print(s)
        s += '\n'
        file_.write(s)

1.0000000000000001e-01  9.1023750000000059e-01  5.9494456301054247e-03  -1.0495230047777873e+00  6.7546093134375944e-03
2.5000000000000000e-01  5.9574999999999945e-01  2.1373023697798019e-02  -1.1880580794534237e+00  1.2292390232929742e-02
5.0000000000000000e-01  4.4700750000000028e-01  2.0609462255228608e-02  -1.2347633725464680e+00  1.9241065540161407e-02
1.0000000000000000e+00  6.6848250000000065e-01  1.2259133767911936e-02  -1.1713393237569372e+00  1.1005499365898393e-02
2.0000000000000000e+00  8.3088999999999946e-01  6.8650346698836415e-03  -1.1347921681050783e+00  7.5696333435353556e-03
3.0000000000000000e+00  8.6187374999999999e-01  6.2811476908316028e-03  -1.1228520128583837e+00  7.4529921795524010e-03
4.0000000000000000e+00  8.6737875000000020e-01  5.8665416039619155e-03  -1.1185051812585218e+00  7.4339146081831529e-03
5.0000000000000000e+00  8.6019874999999968e-01  5.9650538425640850e-03  -1.1233757472859143e+00  7.5655215329441944e-03
6.0000000000000000e+00  8.48823750000000